# 03 — Pinecone Operations

Explore embedding, upserting, and querying with Pinecone Inference API.

**Questions to answer:**
- Does integrated inference (`create_index_for_model`) work for our use case?
- What does retrieval quality look like with real chunks?
- How do we handle the metadata record (zero vector for video info)?
- Latency per operation?

**Output:** Production-ready vectorstore functions → `src/vectorstore.py`

In [1]:
import os
import json
import time
from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Load cached test data
with open("../data/test_transcripts.json") as f:
    test_data = json.load(f)

long_snippets = test_data["long"]["snippets"]

print(f"Connected to Pinecone")
print(f"Existing indexes: {pc.list_indexes().names()}")

Connected to Pinecone
Existing indexes: ['abstractive-question-answering']


## 1. Create index

Using standalone inference (not integrated) because we need to:
- Embed `text` (plain) only, not `text_timestamped`
- Insert zero-vector metadata records (video info, summaries, topics)
- Control exactly what gets embedded

In [2]:
index_name = os.getenv("PINECONE_INDEX_NAME", "askthevideo")

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec={"serverless": {"cloud": "aws", "region": "us-east-1"}},
    )
    print(f"Created index: {index_name}")
    # Wait for index to be ready
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)
    print("Index ready")
else:
    print(f"Index already exists: {index_name}")

index = pc.Index(index_name)
print(index.describe_index_stats())

Created index: askthevideo
Index ready
{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


## 2. Generate chunks from cached data

Import chunking function from notebook 02, generate chunks for the long video.

In [3]:
# Import chunking function (copy from notebook 02 production cell)
# In production this comes from src/chunking.py

def format_time(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}:{m:02d}:{s:02d}"
    return f"{m}:{s:02d}"


def chunk_transcript(snippets, video_id, window_seconds=120, carry_snippets=3):
    if not snippets:
        return []
    chunks = []
    carry = []
    i = 0
    while i < len(snippets):
        chunk_snippets = list(carry)
        window_start = snippets[i]["start"]
        while i < len(snippets) and snippets[i]["start"] < window_start + window_seconds:
            chunk_snippets.append(snippets[i])
            i += 1
        plain_parts = [s["text"].replace("\n", " ") for s in chunk_snippets]
        text = " ".join(plain_parts)
        stamped_lines = [
            f"[{format_time(s['start'])}] {s['text'].replace(chr(10), ' ')}"
            for s in chunk_snippets
        ]
        text_timestamped = "\n".join(stamped_lines)
        start_time = chunk_snippets[0]["start"]
        end_time = chunk_snippets[-1]["start"] + chunk_snippets[-1]["duration"]
        chunks.append({
            "text": text,
            "text_timestamped": text_timestamped,
            "start_time": start_time,
            "end_time": end_time,
            "start_display": format_time(start_time),
            "end_display": format_time(end_time),
            "chunk_index": len(chunks),
            "video_url": f"https://youtu.be/{video_id}?t={int(start_time)}",
        })
        non_carry = chunk_snippets[len(carry):]
        carry = non_carry[-carry_snippets:] if carry_snippets > 0 else []
    return chunks


video_id = "e-gwvmhyU7A"
chunks = chunk_transcript(long_snippets, video_id)
print(f"Generated {len(chunks)} chunks for video {video_id}")
print(f"Sample chunk keys: {list(chunks[0].keys())}")

Generated 90 chunks for video e-gwvmhyU7A
Sample chunk keys: ['text', 'text_timestamped', 'start_time', 'end_time', 'start_display', 'end_display', 'chunk_index', 'video_url']


## 3. Embed chunks via Pinecone Inference API

Standalone embedding with `pc.inference.embed()`. Two input types:
- `"passage"` for documents (chunks being stored)
- `"query"` for search queries (questions at retrieval time)

In [4]:
%%time

# Test with 3 chunks first
test_texts = [c["text"] for c in chunks[:3]]

embeddings = pc.inference.embed(
    model="llama-text-embed-v2",
    inputs=test_texts,
    parameters={"input_type": "passage", "truncate": "END"},
)

print(f"Embeddings returned: {len(embeddings)}")
print(f"Dimension: {len(embeddings[0].values)}")
print(f"First 5 values: {embeddings[0].values[:5]}")

Embeddings returned: 3
Dimension: 1024
First 5 values: [0.0212554931640625, -0.010955810546875, 0.039886474609375, -0.005126953125, 0.04937744140625]
CPU times: user 14.6 ms, sys: 10.3 ms, total: 24.9 ms
Wall time: 585 ms


In [5]:
%%time

BATCH_SIZE = 50  # Pinecone Inference recommends batches of 96 max

vectors = []
for batch_start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[batch_start:batch_start + BATCH_SIZE]
    texts = [c["text"] for c in batch]

    embs = pc.inference.embed(
        model="llama-text-embed-v2",
        inputs=texts,
        parameters={"input_type": "passage", "truncate": "END"},
    )

    for chunk, emb in zip(batch, embs):
        vectors.append({
            "id": f"{video_id}_chunk_{chunk['chunk_index']:03d}",
            "values": emb.values,
            "metadata": {
                "video_id": video_id,
                "text": chunk["text"],
                "text_timestamped": chunk["text_timestamped"],
                "start_time": chunk["start_time"],
                "end_time": chunk["end_time"],
                "start_display": chunk["start_display"],
                "end_display": chunk["end_display"],
                "chunk_index": chunk["chunk_index"],
                "video_url": chunk["video_url"],
            },
        })

    print(f"  Embedded batch {batch_start//BATCH_SIZE + 1}: {len(batch)} chunks")

# Upsert to Pinecone (namespace = video_id)
index.upsert(vectors=vectors, namespace=video_id)

print(f"\nUpserted {len(vectors)} vectors to namespace '{video_id}'")
print(index.describe_index_stats())

  Embedded batch 1: 50 chunks
  Embedded batch 2: 40 chunks

Upserted 90 vectors to namespace 'e-gwvmhyU7A'
{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'e-gwvmhyU7A': {'vector_count': 90}},
 'total_vector_count': 90,
 'vector_type': 'dense'}
CPU times: user 298 ms, sys: 22 ms, total: 320 ms
Wall time: 4.3 s


## 4. Query — retrieval quality test

Test with real questions about the video content. Use `input_type="query"` for search queries.

This is a podcast about AI tools (Perplexity, ChatGPT, etc.) — let's test with questions of varying specificity.

In [7]:
def query_video(question, namespace, top_k=5):
    """Embed a question and query Pinecone. Returns matches."""
    q_emb = pc.inference.embed(
        model="llama-text-embed-v2",
        inputs=[question],
        parameters={"input_type": "query", "truncate": "END"},
    )
    results = index.query(
        vector=q_emb[0].values,
        namespace=namespace,
        top_k=top_k,
        include_metadata=True,
    )
    return results.matches


# Test questions
questions = [
    "What do they think about Perplexity AI?",
    "How does ChatGPT compare to other AI tools?",
    "What is the main topic of this podcast?",
    "Do they discuss pricing of AI tools?",
    "Something completely unrelated like cooking recipes",
]

for q in questions:
    matches = query_video(q, video_id, top_k=3)
    print(f"\nQ: {q}")
    for m in matches:
        print(f"  [{m.score:.3f}] chunk {int(m.metadata['chunk_index']):02d} "
              f"({m.metadata['start_display']}–{m.metadata['end_display']}) "
              f"{m.metadata['text'][:80]}...")


Q: What do they think about Perplexity AI?
  [0.517] chunk 05 (10:02–12:06) - Yeah. - Complete. Okay, answer: "Perplexity AI, while impressive, "is not yet ...
  [0.497] chunk 01 (1:53–4:02) Perplexity is part search engine, part LLM, so how does it work and what role do...
  [0.493] chunk 03 (5:58–8:05) You know, Denis and myself were both academics. Denis is my co-founder and we sa...

Q: How does ChatGPT compare to other AI tools?
  [0.448] chunk 05 (10:02–12:06) - Yeah. - Complete. Okay, answer: "Perplexity AI, while impressive, "is not yet ...
  [0.349] chunk 64 (2:09:31–2:11:38) Right, your eye is so tuned to getting it right. LLMs are fine, like you get the...
  [0.338] chunk 83 (2:48:07–2:50:16) I think it lets you ingest like more detailed version of the pages while answeri...

Q: What is the main topic of this podcast?
  [0.269] chunk 81 (2:44:05–2:46:14) That's exactly what I'm trying to do. - Yeah, and I'm not saying that's the fina...
  [0.226] chunk 89 (3:00:15–3:02:06) 

## 5. Retrieval quality assessment

| Question type | Top score | Quality | Notes |
|---|---|---|---|
| Specific topic (Perplexity) | 0.517 | Good | Top 3 all relevant, correct chunks |
| Comparison (ChatGPT vs others) | 0.448 | OK | Top result relevant, 2-3 are weaker |
| Broad (main topic) | 0.269 | Low | Expected — vague questions match poorly |
| Implied topic (pricing) | 0.279 | OK | No direct pricing discussion in video |
| Irrelevant (cooking) | 0.253 | Correct | Low scores, no false confidence. Hit a barbecue mention. |

Specific questions retrieve well. Broad questions score low — this is where Claude adds value by synthesising across multiple chunks.

## 6. Metadata record (zero vector)

Store video info as a zero-vector record in the same namespace. Used by `get_metadata` tool and for instant reuse detection on re-load.

In [ ]:
#zero_vector = [0.0] * 1024.  # Pinecone now rejects pure zero vectors
zero_vector = [1e-7] + [0.0] * 1023   #near-zero

metadata_record = {
    "id": f"{video_id}_metadata",
    "values": zero_vector,
    "metadata": {
        "type": "metadata",
        "video_id": video_id,
        "video_title": "AI tools podcast (test video)",
        "channel": "Test Channel",
        "duration_seconds": long_snippets[-1]["start"] + long_snippets[-1]["duration"],
        "duration_display": format_time(long_snippets[-1]["start"] + long_snippets[-1]["duration"]),
        "chunk_count": len(chunks),
        "ingested_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    },
}

index.upsert(vectors=[metadata_record], namespace=video_id)

# Verify: fetch by ID (not by query)
fetched = index.fetch(ids=[f"{video_id}_metadata"], namespace=video_id)
meta = fetched.vectors[f"{video_id}_metadata"].metadata
print("Metadata record stored and fetched:")
for k, v in meta.items():
    print(f"  {k}: {v}")

Metadata record stored and fetched:
  channel: Test Channel
  chunk_count: 90.0
  duration_display: 3:02:06
  duration_seconds: 10926.963
  ingested_at: 2026-03-03T14:52:51Z
  type: metadata
  video_id: e-gwvmhyU7A
  video_title: AI tools podcast (test video)


In [10]:
# Query and check that metadata record doesn't appear
matches = query_video("What is this video about?", video_id, top_k=5)

for m in matches:
    record_type = m.metadata.get("type", "chunk")
    print(f"  [{m.score:.3f}] {m.id} (type={record_type})")

has_metadata = any(m.metadata.get("type") == "metadata" for m in matches)
print(f"\nMetadata record in results: {'YES — problem!' if has_metadata else 'No — correct'}")

  [0.215] e-gwvmhyU7A_chunk_075 (type=chunk)
  [0.188] e-gwvmhyU7A_chunk_082 (type=chunk)
  [0.178] e-gwvmhyU7A_chunk_024 (type=chunk)
  [0.142] e-gwvmhyU7A_chunk_088 (type=chunk)
  [0.138] e-gwvmhyU7A_chunk_078 (type=chunk)

Metadata record in results: No — correct


## 7. Latency measurements

In [11]:
import statistics

# Embedding latency (single query)
embed_times = []
for _ in range(5):
    start = time.time()
    pc.inference.embed(
        model="llama-text-embed-v2",
        inputs=["What do they think about AI?"],
        parameters={"input_type": "query", "truncate": "END"},
    )
    embed_times.append(time.time() - start)

# Query latency
query_times = []
test_emb = pc.inference.embed(
    model="llama-text-embed-v2",
    inputs=["test query"],
    parameters={"input_type": "query", "truncate": "END"},
)
for _ in range(5):
    start = time.time()
    index.query(vector=test_emb[0].values, namespace=video_id, top_k=5, include_metadata=True)
    query_times.append(time.time() - start)

# Fetch by ID latency
fetch_times = []
for _ in range(5):
    start = time.time()
    index.fetch(ids=[f"{video_id}_metadata"], namespace=video_id)
    fetch_times.append(time.time() - start)

def stats(times, label):
    print(f"{label}: median={statistics.median(times)*1000:.0f}ms, "
          f"min={min(times)*1000:.0f}ms, max={max(times)*1000:.0f}ms")

stats(embed_times, "Embed (single query)")
stats(query_times, "Query (top_k=5)    ")
stats(fetch_times, "Fetch by ID        ")

print(f"\nEnd-to-end estimate (embed + query): "
      f"~{(statistics.median(embed_times) + statistics.median(query_times))*1000:.0f}ms")

Embed (single query): median=499ms, min=231ms, max=520ms
Query (top_k=5)    : median=176ms, min=140ms, max=1001ms
Fetch by ID        : median=178ms, min=136ms, max=185ms

End-to-end estimate (embed + query): ~675ms


## 8. Latency summary

| Operation | Median | Notes |
|---|---|---|
| Embed (single query) | ~500ms | Pinecone Inference API |
| Query (top_k=5) | ~175ms | Serverless, cosine similarity |
| Fetch by ID | ~175ms | Direct lookup, metadata records |
| End-to-end (embed+query) | ~675ms | Below 1s — good UX |
| Batch embed (90 chunks) | ~4.3s | One-time at ingestion |

The 1s max spike on query is likely cold-start or network jitter. Median is consistent.

## 9. Production functions

In [12]:
# @export src/vectorstore.py
from pinecone import Pinecone
import os
import time

EMBED_MODEL = "llama-text-embed-v2"
EMBED_BATCH_SIZE = 50
SENTINEL_VECTOR = [1e-7] + [0.0] * 1023


def get_pinecone_index():
    """Connect to Pinecone and return the index."""
    pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
    index_name = os.getenv("PINECONE_INDEX_NAME", "askthevideo")
    return pc, pc.Index(index_name)


def embed_texts(pc, texts: list[str], input_type: str = "passage") -> list[list[float]]:
    """Embed a list of texts via Pinecone Inference API.

    Args:
        pc: Pinecone client
        texts: list of strings to embed
        input_type: "passage" for documents, "query" for search queries
    Returns:
        list of embedding vectors (list of floats)
    """
    all_embeddings = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i:i + EMBED_BATCH_SIZE]
        embs = pc.inference.embed(
            model=EMBED_MODEL,
            inputs=batch,
            parameters={"input_type": input_type, "truncate": "END"},
        )
        all_embeddings.extend([e.values for e in embs])
    return all_embeddings


def upsert_chunks(pc, index, chunks: list[dict], video_id: str) -> int:
    """Embed and upsert transcript chunks into Pinecone.

    Args:
        pc: Pinecone client
        index: Pinecone index
        chunks: list of chunk dicts from chunk_transcript()
        video_id: YouTube video ID (used as namespace)
    Returns:
        number of vectors upserted
    """
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(pc, texts, input_type="passage")

    vectors = []
    for chunk, emb in zip(chunks, embeddings):
        vectors.append({
            "id": f"{video_id}_chunk_{chunk['chunk_index']:03d}",
            "values": emb,
            "metadata": {
                "video_id": video_id,
                "type": "chunk",
                "text": chunk["text"],
                "text_timestamped": chunk["text_timestamped"],
                "start_time": chunk["start_time"],
                "end_time": chunk["end_time"],
                "start_display": chunk["start_display"],
                "end_display": chunk["end_display"],
                "chunk_index": chunk["chunk_index"],
                "video_url": chunk["video_url"],
            },
        })

    index.upsert(vectors=vectors, namespace=video_id)
    return len(vectors)


def upsert_metadata_record(index, video_id: str, metadata: dict):
    """Store a metadata record (sentinel vector) in the video namespace.

    Args:
        index: Pinecone index
        video_id: YouTube video ID
        metadata: dict with video_title, channel, duration_seconds, etc.
    """
    record = {
        "id": f"{video_id}_metadata",
        "values": SENTINEL_VECTOR,
        "metadata": {
            "type": "metadata",
            "video_id": video_id,
            **metadata,
            "ingested_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        },
    }
    index.upsert(vectors=[record], namespace=video_id)


def query_chunks(pc, index, question: str, video_id: str, top_k: int = 5) -> list[dict]:
    """Embed a question and retrieve matching chunks.

    Args:
        pc: Pinecone client
        index: Pinecone index
        question: user's question
        video_id: namespace to search
        top_k: number of results
    Returns:
        list of dicts with score + metadata
    """
    emb = embed_texts(pc, [question], input_type="query")[0]
    results = index.query(
        vector=emb,
        namespace=video_id,
        top_k=top_k,
        include_metadata=True,
    )
    return [
        {"score": m.score, "id": m.id, **m.metadata}
        for m in results.matches
        if m.metadata.get("type") != "metadata"
    ]


def fetch_metadata(index, video_id: str) -> dict | None:
    """Fetch the metadata record for a video. Returns None if not found."""
    result = index.fetch(ids=[f"{video_id}_metadata"], namespace=video_id)
    vec = result.vectors.get(f"{video_id}_metadata")
    if vec:
        return dict(vec.metadata)
    return None


def namespace_exists(index, video_id: str) -> bool:
    """Check if a video namespace already has vectors."""
    stats = index.describe_index_stats()
    return video_id in stats.namespaces

In [13]:
# Test against already-upserted data

# 1. namespace_exists
print(f"namespace_exists('{video_id}'): {namespace_exists(index, video_id)}")
print(f"namespace_exists('fake_id'): {namespace_exists(index, 'fake_id')}")

# 2. fetch_metadata
meta = fetch_metadata(index, video_id)
print(f"\nfetch_metadata: {meta['video_title']} ({meta['duration_display']})")

# 3. query_chunks
results = query_chunks(pc, index, "What do they say about Perplexity?", video_id, top_k=3)
print(f"\nquery_chunks (top 3):")
for r in results:
    print(f"  [{r['score']:.3f}] chunk {int(r['chunk_index']):02d} "
          f"({r['start_display']}–{r['end_display']})")

# 4. Verify no metadata record leaks into query results
has_meta = any(r.get("type") == "metadata" for r in results)
print(f"\n✅ All functions work" if not has_meta else "❌ Metadata leak!")

namespace_exists('e-gwvmhyU7A'): True
namespace_exists('fake_id'): False

fetch_metadata: AI tools podcast (test video) (3:02:06)

query_chunks (top 3):
  [0.456] chunk 01 (1:53–4:02)
  [0.451] chunk 04 (7:59–10:06)
  [0.423] chunk 03 (5:58–8:05)

✅ All functions work


## Summary

**What we built:**
- `get_pinecone_index()` — connect to Pinecone, return client + index
- `embed_texts(pc, texts, input_type)` — batch embed via Pinecone Inference API
- `upsert_chunks(pc, index, chunks, video_id)` — embed + upsert transcript chunks
- `upsert_metadata_record(index, video_id, metadata)` — store video info as sentinel-vector record
- `query_chunks(pc, index, question, video_id, top_k)` — embed question + retrieve matching chunks (filters out metadata records)
- `fetch_metadata(index, video_id)` — direct ID lookup for video info
- `namespace_exists(index, video_id)` — check if video already ingested

**Production cells tagged:** 1 cell → `src/vectorstore.py`

**Key decisions:**

| Decision | Choice | Reasoning |
|---|---|---|
| Index creation | Standalone `create_index` | Need sentinel-vector records (metadata, summary, topics) that must not be auto-embedded |
| Embedding approach | `pc.inference.embed()` standalone | Full control over what gets embedded, separate passage/query input types |
| Metadata records | Sentinel vector `[1e-7, 0, 0, ...]` | Pinecone rejects pure zero vectors. Near-zero never appears in query results |
| Metadata filtering | `type` field in metadata + post-query filter | Sentinel records score near-zero but filtered out as safety net |
| Batch size | 50 per embed call | Conservative, within Pinecone's 96 max recommendation |

**Latency (median):**

| Operation | Median | 
|---|---|
| Embed single query | ~500ms |
| Query top_k=5 | ~175ms |
| Fetch by ID | ~175ms |
| End-to-end (embed+query) | ~675ms |
| Batch embed+upsert (90 chunks) | ~4.3s |

**Retrieval quality:**
- Specific questions: strong (0.45-0.52 top score)
- Comparison questions: decent (0.45 top)
- Broad/vague questions: low (0.22-0.27) — expected, Claude synthesises across chunks
- Irrelevant questions: low scores, no false confidence

**Next:** Notebook 04 — Claude Tools (vector_search, summarize_video, get_topics, compare_videos, get_metadata)